In [1]:
import os
print(os.getcwd())

c:\Users\vp532\OneDrive\Desktop\Mini_project-2


In [2]:
import pandas as pd
load  = pd.read_csv('datasets/processed/load_scaled.csv')
solar = pd.read_csv('datasets/processed/solar_scaled.csv')
print(load.shape, solar.shape)
print(load.columns.tolist())
print(solar.columns.tolist())

(720, 1) (720, 1)
['load_mw']
['p_solar_mw']


In [4]:
from data_load import prepare_load_series
from data_solar import prepare_solar_series
from battery_model import BatteryModel
from grid_simulator import create_network, run_load_flow
from label_generator import generate_label, get_severity, get_blackout_margin
print("All imports OK")

All imports OK


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('datasets/processed/simulation_results.csv')

print(f"Shape        : {df.shape}")
print(f"Columns      : {df.columns.tolist()}")
print(f"\nBlackout distribution:")
print(df['blackout'].value_counts())
print(f"\nFeature stats:")
print(df.describe().to_string())
print(f"\nMissing values:")
print(df.isnull().sum())

Shape        : (720, 18)
Columns      : ['timestep', 'day', 'hour_of_day', 'is_night', 'p_load_mw', 'p_solar_mw', 'p_battery_mw', 'p_grid_mw', 'soc', 'v_min', 'v_max', 'v_mean', 'line_loading_max', 'p_loss_mw', 'converged', 'v_margin', 'severity', 'blackout']

Blackout distribution:
blackout
0    566
1    154
Name: count, dtype: int64

Feature stats:
         timestep         day  hour_of_day    is_night   p_load_mw  p_solar_mw  p_battery_mw   p_grid_mw         soc       v_min       v_max      v_mean  line_loading_max   p_loss_mw  converged    v_margin    blackout
count  720.000000  720.000000   720.000000  720.000000  720.000000  720.000000    720.000000  720.000000  720.000000  720.000000  720.000000  720.000000        720.000000  720.000000      720.0  720.000000  720.000000
mean   359.500000   14.500000    11.500000    0.416667    2.801731    0.723704      0.000218    2.077809    0.102825    0.946710    1.016688    0.975317         33.441959    0.134244        1.0    0.016710    0.

In [1]:
import pandapower as pp
import pandapower.networks as pn

print("Pandapower installed successfully!")

Pandapower installed successfully!


In [2]:
pip show pandapower

Name: pandapower
Version: 3.4.0
Summary: An easy to use open source tool for power system modeling, analysis and optimization with a high degree of automation.
Home-page: https://www.pandapower.org
Author: 
Author-email: Leon Thurner <leon.thurner@retoflow.de>, Alexander Scheidler <alexander.scheidler@iee.fraunhofer.de>, Mike Vogt <mike.vogt@iee.fraunhofer.de>
License: 
Location: C:\Users\vp532\AppData\Roaming\Python\Python312\site-packages
Requires: deepdiff, geojson, networkx, numpy, packaging, pandas, pandera, scipy, tqdm, typing_extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandapower.networks as pn
net = pn.case33bw()
print(net.line[['from_bus', 'to_bus', 'max_i_ka']].to_string())

    from_bus  to_bus  max_i_ka
0          0       1   99999.0
1          1       2   99999.0
2          2       3   99999.0
3          3       4   99999.0
4          4       5   99999.0
5          5       6   99999.0
6          6       7   99999.0
7          7       8   99999.0
8          8       9   99999.0
9          9      10   99999.0
10        10      11   99999.0
11        11      12   99999.0
12        12      13   99999.0
13        13      14   99999.0
14        14      15   99999.0
15        15      16   99999.0
16        16      17   99999.0
17         1      18   99999.0
18        18      19   99999.0
19        19      20   99999.0
20        20      21   99999.0
21         2      22   99999.0
22        22      23   99999.0
23        23      24   99999.0
24         5      25   99999.0
25        25      26   99999.0
26        26      27   99999.0
27        27      28   99999.0
28        28      29   99999.0
29        29      30   99999.0
30        30      31   99999.0
31      

In [4]:
import pandapower as pp
import pandapower.networks as pn

net = pn.case33bw()
print(net)
print(net.bus)
print(net.load)

This pandapower network includes the following parameter tables:
   - bus (33 elements)
   - load (32 elements)
   - ext_grid (1 element)
   - line (37 elements)
   - poly_cost (1 element)
   name  vn_kv type zone  in_service  max_vm_pu  min_vm_pu  \
0     0  12.66    b  1.0        True        1.0        1.0   
1     1  12.66    b  1.0        True        1.1        0.9   
2     2  12.66    b  1.0        True        1.1        0.9   
3     3  12.66    b  1.0        True        1.1        0.9   
4     4  12.66    b  1.0        True        1.1        0.9   
5     5  12.66    b  1.0        True        1.1        0.9   
6     6  12.66    b  1.0        True        1.1        0.9   
7     7  12.66    b  1.0        True        1.1        0.9   
8     8  12.66    b  1.0        True        1.1        0.9   
9     9  12.66    b  1.0        True        1.1        0.9   
10   10  12.66    b  1.0        True        1.1        0.9   
11   11  12.66    b  1.0        True        1.1        0.9   
12   

In [4]:
import pandapower.networks as pn

net = pn.case33bw()
print("Load table:")
print(net.load[['bus', 'p_mw', 'q_mvar']].to_string())
print(f"\nTotal base load: {net.load['p_mw'].sum():.4f} MW")
print(f"Total base qload: {net.load['q_mvar'].sum():.4f} MVAr")
print(f"\nExt grid bus: {net.ext_grid['bus'].values}")
print(f"Number of lines: {len(net.line)}")

Load table:
    bus   p_mw  q_mvar
0     1  0.100   0.060
1     2  0.090   0.040
2     3  0.120   0.080
3     4  0.060   0.030
4     5  0.060   0.020
5     6  0.200   0.100
6     7  0.200   0.100
7     8  0.060   0.020
8     9  0.060   0.020
9    10  0.045   0.030
10   11  0.060   0.035
11   12  0.060   0.035
12   13  0.120   0.080
13   14  0.060   0.010
14   15  0.060   0.020
15   16  0.060   0.020
16   17  0.090   0.040
17   18  0.090   0.040
18   19  0.090   0.040
19   20  0.090   0.040
20   21  0.090   0.040
21   22  0.090   0.050
22   23  0.420   0.200
23   24  0.420   0.200
24   25  0.060   0.025
25   26  0.060   0.025
26   27  0.060   0.020
27   28  0.120   0.070
28   29  0.200   0.600
29   30  0.150   0.070
30   31  0.210   0.100
31   32  0.060   0.040

Total base load: 3.7150 MW
Total base qload: 2.3000 MVAr

Ext grid bus: [0]
Number of lines: 37


In [1]:
import pandas as pd
df = pd.read_csv('datasets/processed/lf_lookup.csv')
no_solar = df[df['p_solar_mw'] == 0.0].sort_values('net_load_mw')
threshold = df[df['p_solar_mw'] == 0.0]
below = threshold[threshold['v_min'] < 0.93]['net_load_mw']
above = threshold[threshold['v_min'] >= 0.93]['net_load_mw']
print(f"Blackout above net_load : {below.min():.3f} MW")
print(f"Safe below net_load     : {above.max():.3f} MW")
print(f"Battery needed to fix   : {below.min() - above.max():.3f} MW")

Blackout above net_load : 3.125 MW
Safe below net_load     : 2.979 MW
Battery needed to fix   : 0.146 MW


In [2]:
import pandas as pd
import numpy as np

load = pd.read_csv('datasets/processed/load_scaled.csv')['load_mw']
solar = pd.read_csv('datasets/processed/solar_scaled.csv')['p_solar_mw']

# Net load after solar (no battery)
net_load = load - solar
net_load = net_load.clip(lower=0)

rescuable = ((net_load >= 3.125) & (net_load <= 3.375)).sum()
too_high  = (net_load > 3.375).sum()
safe      = (net_load < 3.125).sum()

print(f"Safe hours      : {safe}   (no blackout possible)")
print(f"Rescuable hours : {rescuable}   (battery CAN fix these)")
print(f"Unrescuable     : {too_high}   (load too high, battery too small)")
print(f"Total           : {len(net_load)}")

Safe hours      : 583   (no blackout possible)
Rescuable hours : 39   (battery CAN fix these)
Unrescuable     : 98   (load too high, battery too small)
Total           : 720


In [2]:
import pandapower.networks as pn

net = pn.case33bw()

In [3]:
print("Number of buses:", len(net.bus))
print("Number of lines:", len(net.line))
print("Number of loads:", len(net.load))

Number of buses: 33
Number of lines: 37
Number of loads: 32


In [4]:
import pandapower as pp

pp.runpp(net)

print(net.res_bus[['vm_pu']].head())

      vm_pu
0  1.000000
1  0.997032
2  0.982938
3  0.975456
4  0.968059


In [5]:
total_base_load = net.load.p_mw.sum()
print("Total Base Load (MW):", total_base_load)

Total Base Load (MW): 3.715


Pbase​=3.715 MW

In [6]:
print("Min Voltage:", net.res_bus.vm_pu.min())
print("Max Voltage:", net.res_bus.vm_pu.max())

Min Voltage: 0.9130904822858367
Max Voltage: 1.0


In [7]:
net.load[['bus', 'p_mw']].head()

,bus,p_mw
0,1,0.10
1,2,0.09
2,3,0.12
3,4,0.06
4,5,0.06
